In [49]:
import pandas as pd
from huggingface_hub import hf_hub_download

def load_table(filename):
    path = hf_hub_download(repo_id="hao-li/AIDev", filename=filename, repo_type="dataset")
    return pd.read_parquet(path)

prs      = load_table("pull_request.parquet")
commits  = load_table("pr_commits.parquet")
reviews  = load_table("pr_reviews.parquet")
rev_cmts = load_table("pr_review_comments_v2.parquet")
timeline = load_table("pr_timeline.parquet")

In [50]:
import sys
sys.path.insert(0, '.')
from helpers import build_pr_states, categorize

merged = prs[prs['merged_at'].notna()].copy()

pr_states = build_pr_states(reviews, merged['id'])

merged['review_category'] = merged['id'].apply(lambda pr_id: categorize(pr_id, pr_states))

revised_prs = merged[
    merged['review_category'].isin([
        'changes_requested_then_approved',
        'changes_requested_and_dismissed_then_approved'
    ])
].copy()

print(f"Problematic-Revised PRs (raw): {len(revised_prs):,}")
revised_prs['review_category'].value_counts()

Problematic-Revised PRs (raw): 427


review_category
changes_requested_then_approved                  386
changes_requested_and_dismissed_then_approved     41
Name: count, dtype: int64

In [51]:
# Filter 1: Remove self-approvals — APPROVED review must be from a human User
human_approved_pr_ids = set(
    reviews[
        (reviews['state'] == 'APPROVED') &
        (reviews['user_type'] == 'User')
    ]['pr_id'].unique()
)

revised_prs = revised_prs[revised_prs['id'].isin(human_approved_pr_ids)]
print(f"After removing bot-approved PRs: {len(revised_prs):,}")

After removing bot-approved PRs: 403


In [52]:
# Filter 2: Remove PRs where CHANGES_REQUESTED came only from bots
human_cr_pr_ids = set(
    reviews[
        (reviews['state'] == 'CHANGES_REQUESTED') &
        (reviews['user_type'] == 'User')
    ]['pr_id'].unique()
)

revised_prs = revised_prs[revised_prs['id'].isin(human_cr_pr_ids)]
print(f"After requiring human CHANGES_REQUESTED: {len(revised_prs):,}")

After requiring human CHANGES_REQUESTED: 397


In [53]:
pr_reviews_sub = reviews[
    reviews['pr_id'].isin(revised_prs['id'])
][['pr_id', 'id', 'user', 'user_type', 'state', 'submitted_at']].copy()

pr_commits_sub = commits[
    commits['pr_id'].isin(revised_prs['id'])
][['pr_id', 'sha', 'author', 'message']].copy()

print(f"Reviews (on revised PRs): {len(pr_reviews_sub):,}")
print(f"Commits (on revised PRs): {len(pr_commits_sub):,}")

Reviews (on revised PRs): 3,079
Commits (on revised PRs): 3,107


In [54]:
# Pull inline review comments from CHANGES_REQUESTED reviews
cr_review_ids = set(
    reviews[
        (reviews['pr_id'].isin(revised_prs['id'])) &
        (reviews['state'] == 'CHANGES_REQUESTED') &
        (reviews['user_type'] == 'User')
    ]['id'].unique()
)

cr_inline_comments = rev_cmts[
    rev_cmts['pull_request_review_id'].isin(cr_review_ids)
][['pull_request_review_id', 'user', 'path', 'diff_hunk', 'body', 'created_at']].copy()

# Join back to pr_id
review_id_to_pr = reviews.set_index('id')['pr_id'].to_dict()
cr_inline_comments['pr_id'] = cr_inline_comments['pull_request_review_id'].map(review_id_to_pr)

print(f"PRs with inline CR comments:  {cr_inline_comments['pr_id'].nunique():,}")
print(f"Total inline CR comments:     {len(cr_inline_comments):,}")

PRs with inline CR comments:  300
Total inline CR comments:     1,107


In [55]:
print("=== Problematic-Revised Clean Subset ===")
print(f"Total PRs:                          {len(revised_prs):,}")
print(f"  changes_requested_then_approved:  {(revised_prs['review_category'] == 'changes_requested_then_approved').sum():,}")
print(f"  CR + dismissed then approved:     {(revised_prs['review_category'] == 'changes_requested_and_dismissed_then_approved').sum():,}")
print(f"PRs with inline CR comments:        {cr_inline_comments['pr_id'].nunique():,}")

=== Problematic-Revised Clean Subset ===
Total PRs:                          397
  changes_requested_then_approved:  364
  CR + dismissed then approved:     33
PRs with inline CR comments:        300


In [ ]:
commit_counts = pr_commits_sub.groupby('pr_id')['sha'].count().rename('commit_count')
revised_prs = revised_prs.merge(commit_counts, left_on='id', right_index=True, how='left')

non_squashed = revised_prs[revised_prs['commit_count'] > 1]

print(f"Revised PRs (total):         {len(revised_prs):,}")
print(f"Non-squashed (>1 commit):    {len(non_squashed):,}")
print(f"Likely squashed (1 commit):  {(revised_prs['commit_count'] == 1).sum():,}")
print()
print(non_squashed['commit_count'].describe())

In [56]:
sample_id = revised_prs.sample(1, random_state=42)['id'].iloc[0]
pr_html_url = prs.loc[prs['id'] == sample_id, 'html_url'].iloc[0]

print("=== PR ===")
print(prs[prs['id'] == sample_id][['id', 'title', 'agent', 'repo_url', 'html_url', 'created_at', 'merged_at']].to_string(index=False))

print("\n=== Reviews ===")
print(
    pr_reviews_sub[pr_reviews_sub['pr_id'] == sample_id]
    .sort_values('submitted_at')[['state', 'user', 'user_type', 'submitted_at']]
    .to_string(index=False)
)

print("\n=== Commits ===")
commits_sample = pr_commits_sub[pr_commits_sub['pr_id'] == sample_id][['sha', 'author', 'message']].copy()
commits_sample['url'] = pr_html_url + '/commits/' + commits_sample['sha']
print(commits_sample.to_string(index=False))

print("\n=== Inline CR Comments ===")
print(
    cr_inline_comments[cr_inline_comments['pr_id'] == sample_id]
    [['path', 'body']]
    .to_string(index=False)
)

=== PR ===
        id                                                   title   agent                                      repo_url                                      html_url           created_at            merged_at
3121678248 Don't show code lenses for code with compilation errors Copilot https://api.github.com/repos/microsoft/qsharp https://github.com/microsoft/qsharp/pull/2511 2025-06-05T15:25:30Z 2025-06-12T20:38:27Z

=== Reviews ===
            state                   user user_type         submitted_at
CHANGES_REQUESTED             minestarks      User 2025-06-05T15:48:29Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-06-05T16:10:22Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-06-05T16:10:29Z
         APPROVED             minestarks      User 2025-06-09T16:50:12Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-06-09T17:05:06Z
        COMMENTED copilot-swe-agent[bot]       Bot 2025-06-09T17:05:13Z
        COMMENTED                 idavis      User